In [ ]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import math
import random
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:0.4f}")

DATA_PATH = "Q-RMR Data.xlsx"
FEATURE_COLS = ["RQD", "Jn", "Jr", "Ja", "Jw", "SRF"]
TARGET_COL = "RMR"

SEED = 0
XGB_RANDOM_STATE = 437

POPULATION_SIZES = [20, 50, 100, 150, 200]
MAX_ITER = 300
G0 = 1.0
ALPHA = 20
GMIN = 1e-4
EPS = 1e-12

BOUNDS = {
    "n_estimators": (100, 1000),
    "learning_rate": (0.01, 0.30),
    "max_depth": (2, 10)
}

df = pd.read_excel(DATA_PATH)
df = df[FEATURE_COLS + [TARGET_COL]].copy()
df[FEATURE_COLS + [TARGET_COL]] = df[FEATURE_COLS + [TARGET_COL]].apply(pd.to_numeric, errors="coerce")
df = df.dropna().reset_index(drop=True)

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, shuffle=True
)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, shuffle=True
)

print(f"Dataset = {DATA_PATH}")
print(f"Total samples = {len(df)}")
print(f"Train / Validation / Test = {len(y_tr)} / {len(y_val)} / {len(y_te)}")

In [ ]:
def calc_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae

def decode_position(position):
    n_estimators = int(round(position[0]))
    learning_rate = float(position[1])
    max_depth = int(round(position[2]))

    n_estimators = max(BOUNDS["n_estimators"][0], min(BOUNDS["n_estimators"][1], n_estimators))
    learning_rate = max(BOUNDS["learning_rate"][0], min(BOUNDS["learning_rate"][1], learning_rate))
    max_depth = max(BOUNDS["max_depth"][0], min(BOUNDS["max_depth"][1], max_depth))

    return {
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "max_depth": max_depth
    }

def build_xgb(params):
    return xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        min_child_weight=1,
        gamma=0,
        subsample=1.0,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        tree_method="hist",
        verbosity=0,
        n_jobs=1,
        random_state=XGB_RANDOM_STATE
    )

def fitness_function(position):
    params = decode_position(position)
    model = build_xgb(params)
    model.fit(X_tr, y_tr)
    y_pred_val = model.predict(X_val)
    rmse_val = math.sqrt(mean_squared_error(y_val, y_pred_val))
    return rmse_val

class OGSAXGB:
    def __init__(self, pop_size, max_iter, g0, alpha, gmin, seed):
        self.pop_size = pop_size
        self.max_iter = max_iter
        self.g0 = g0
        self.alpha = alpha
        self.gmin = gmin
        self.seed = seed

        random.seed(seed)
        np.random.seed(seed)

        self.lb = np.array([BOUNDS[k][0] for k in BOUNDS], dtype=float)
        self.ub = np.array([BOUNDS[k][1] for k in BOUNDS], dtype=float)
        self.dim = len(BOUNDS)

    def initialize_population(self):
        pop = np.random.uniform(self.lb, self.ub, size=(self.pop_size, self.dim))
        opp = self.lb + self.ub - pop
        combined = np.vstack([pop, opp])
        fitness = np.array([fitness_function(ind) for ind in combined])
        best_idx = np.argsort(fitness)[:self.pop_size]
        return combined[best_idx].copy(), fitness[best_idx].copy()

    def gravitational_constant(self, t):
        g = self.g0 * np.exp(-self.alpha * t / self.max_iter)
        return max(g, self.gmin)

    def mass_calculation(self, fitness):
        best = np.min(fitness)
        worst = np.max(fitness)

        if abs(worst - best) < EPS:
            return np.ones_like(fitness) / len(fitness)

        q = (fitness - worst) / (best - worst + EPS)
        q = np.maximum(q, 0)
        return q / (np.sum(q) + EPS)

    def optimize(self):
        positions, fitness = self.initialize_population()
        velocities = np.zeros_like(positions)

        best_idx = np.argmin(fitness)
        global_best_position = positions[best_idx].copy()
        global_best_fitness = fitness[best_idx]

        history = []

        for t in range(self.max_iter):
            masses = self.mass_calculation(fitness)
            G = self.gravitational_constant(t + 1)
            accelerations = np.zeros_like(positions)

            for i in range(self.pop_size):
                force = np.zeros(self.dim)
                for j in range(self.pop_size):
                    if i == j:
                        continue
                    dist = np.linalg.norm(positions[j] - positions[i]) + EPS
                    rand_vec = np.random.rand(self.dim)
                    force += rand_vec * G * (masses[i] * masses[j]) * (positions[j] - positions[i]) / dist
                accelerations[i] = force / (masses[i] + EPS)

            velocities = np.random.rand(self.pop_size, self.dim) * velocities + accelerations
            positions = positions + velocities
            positions = np.clip(positions, self.lb, self.ub)

            current_fitness = np.array([fitness_function(ind) for ind in positions])

            opp_positions = self.lb + self.ub - positions
            opp_positions = np.clip(opp_positions, self.lb, self.ub)
            opp_fitness = np.array([fitness_function(ind) for ind in opp_positions])

            improve_mask = opp_fitness < current_fitness
            positions[improve_mask] = opp_positions[improve_mask]
            current_fitness[improve_mask] = opp_fitness[improve_mask]

            fitness = current_fitness

            best_idx = np.argmin(fitness)
            if fitness[best_idx] < global_best_fitness:
                global_best_fitness = fitness[best_idx]
                global_best_position = positions[best_idx].copy()

            history.append(global_best_fitness)

        return {
            "best_position": global_best_position,
            "best_fitness": global_best_fitness,
            "best_params": decode_position(global_best_position),
            "history": history
        }

In [ ]:
all_results = []

for pop_size in POPULATION_SIZES:
    optimizer = OGSAXGB(
        pop_size=pop_size,
        max_iter=MAX_ITER,
        g0=G0,
        alpha=ALPHA,
        gmin=GMIN,
        seed=SEED
    )

    result = optimizer.optimize()
    result["population_size"] = pop_size
    all_results.append(result)

summary_rows = []
for res in all_results:
    summary_rows.append([
        res["population_size"],
        res["best_params"]["n_estimators"],
        res["best_params"]["learning_rate"],
        res["best_params"]["max_depth"],
        res["best_fitness"]
    ])

summary_df = pd.DataFrame(
    summary_rows,
    columns=["Population_size", "Best_n_estimators", "Best_learning_rate", "Best_max_depth", "Best_val_RMSE"]
).sort_values(by="Best_val_RMSE", ascending=True).reset_index(drop=True)

print("Population comparison summary:")
display(summary_df)

pop_compare_df = pd.DataFrame({
    "Iteration": np.arange(1, MAX_ITER + 1),
    "Population-20":  next(res["history"] for res in all_results if res["population_size"] == 20),
    "Population-50":  next(res["history"] for res in all_results if res["population_size"] == 50),
    "Population-100": next(res["history"] for res in all_results if res["population_size"] == 100),
    "Population-150": next(res["history"] for res in all_results if res["population_size"] == 150),
    "Population-200": next(res["history"] for res in all_results if res["population_size"] == 200),
})

pop_compare_df.to_excel("Pop_Compare.xlsx", index=False)
print("Pop_Compare.xlsx has been exported successfully.")

best_result = min(all_results, key=lambda x: x["best_fitness"])
best_params = best_result["best_params"]

final_model = build_xgb(best_params)
final_model.fit(X_tr, y_tr)

yhat_tr = final_model.predict(X_tr)
yhat_val = final_model.predict(X_val)
yhat_te = final_model.predict(X_te)

r2_tr, rmse_tr, mae_tr = calc_metrics(y_tr, yhat_tr)
r2_val, rmse_val, mae_val = calc_metrics(y_val, yhat_val)
r2_te, rmse_te, mae_te = calc_metrics(y_te, yhat_te)

final_metrics_df = pd.DataFrame({
    "Subset": ["Train", "Validation", "Test"],
    "R2": [r2_tr, r2_val, r2_te],
    "RMSE": [rmse_tr, rmse_val, rmse_te],
    "MAE": [mae_tr, mae_val, mae_te]
})

print("Best OGSA-XGBoost parameters:")
print(best_params)
print(f"Best population size: {best_result['population_size']}")
print(f"Best validation RMSE during optimization: {best_result['best_fitness']:.4f}")

print("\nFinal performance:")
display(final_metrics_df)